# In this file we do some testing with a new approach for deriving probability guarantees that circumvents the multiple hypothesis testing issue, without requiring a 3rd test set

## Setup

$m$ : number of solutions for which we want to compute probability guarantees \
$N$ : number of data points (size of test set) \
$K$ : number of samples we use to estimate covariance matrix \

For this file we will assume that $m=2$, $N=100$ and $K=10,000$.

The random variables under consideration are: $\left( \mathbb{1}_{\left[ f(\mathbf{x}_1, \mathbf{z}) \leq 0 \right]}, \dots, \mathbb{1}_{\left[ f(\mathbf{x}_m, \mathbf{z}) \leq 0 \right]} \right)$.

We evaluate two solutions $\mathbf{x}_1$ and $\mathbf{x}_2$. We assume that the true distributions of $\mathbb{1}_{\left[ f(\mathbf{x}_1, \mathbf{z}) \leq 0 \right]} \sim$ Bernoulli with $p=0.7$ and $\mathbb{1}_{\left[ f(\mathbf{x}_2, \mathbf{z}) \leq 0 \right]} \sim$ Bernoulli with $p=0.8$. Furthermore we assume that these are independent.

In [13]:
import math
import numpy as np

In [21]:
# true probabilities that x1 and x2 are feasible
p_x1 = 0.7
p_x2 = 0.8
print(f"true prob: p1={p_x1}, p2={p_x2}")

true prob: p1=0.7, p2=0.8


In [22]:
# true std deviations 
sigma_x1 = math.sqrt(p_x1*(1-p_x1))
sigma_x2 = math.sqrt(p_x2*(1-p_x2))
print(f"true stdev: sigma1={sigma_x1}, sigma2={sigma_x2}")

true stdev: sigma1=0.45825756949558405, sigma2=0.39999999999999997


Now, let's introduce the data. Assume we have $N=100$ observations of $\mathbb{1}_{\left[ f(\mathbf{x}_1, \mathbf{z}) \leq 0 \right]}$ and $\mathbb{1}_{\left[ f(\mathbf{x}_2, \mathbf{z}) \leq 0 \right]}$ 

In [25]:
N = 100
seed = 123456
rng = np.random.default_rng(seed)
data_points_x1 = rng.binomial(n=1, p=p_x1, size=N)
data_points_x2 = rng.binomial(n=1, p=p_x2, size=N)

In [24]:
p_hat_x1 = np.mean(data_points_x1)
p_hat_x2 = np.mean(data_points_x2)
print(f"empirical prob: p1={p_hat_x1}, p2={p_hat_x2}")

empirical prob: p1=0.65, p2=0.85


In [26]:
sigma_hat_x1 = np.std(data_points_x1)
sigma_hat_x2 = np.std(data_points_x2)
print(f"empirical stdev: sigma1={sigma_hat_x1}, sigma2={sigma_hat_x2}")

empirical stdev: sigma1=0.4769696007084728, sigma2=0.3570714214271425


## Computing the feasibility certificates / probability guarantees

We compute the feasibility certificates $\gamma_i$ as follows: $$\gamma_i = \hat{p}_i - \sqrt{\frac{r}{N}} \sqrt{\hat{\sigma_i}}, \qquad i=\mathbf{x}_1, \dots, \mathbf{x}_m.$$ To do this we first need to derive $\sqrt{r}$, which should be set equal to the $(1-\alpha)$ quantile of the distribution of $\max_{i=1,\dots,m} Y$, where $Y$ is (a centered transformation) of the random vector $\left( \frac{1}{\sigma_{\mathbf{x}_1}} \mathbb{1}_{\left[ f(\mathbf{x}_1, \mathbf{z}) \leq 0 \right]}, \dots, \frac{1}{\sigma_{\mathbf{x}_m}} \mathbb{1}_{\left[ f(\mathbf{x}_m, \mathbf{z}) \leq 0 \right]} \right)$. We do not know the true distribution of $Y$, but we can determine a consistent estimator of the quantile via estimating the covariance matrix using our data and then performing some empirical sampling of $Y$.

In [30]:
# first we estimate the correlation coefficient matrix using our data_points
Sigma_hat = np.corrcoef(data_points_x1 , data_points_x2)
Sigma_hat

array([[ 1.        , -0.07339462],
       [-0.07339462,  1.        ]])

This matrix $\hat{\Sigma}$ is an estimate of $\Sigma$ (the covariance matrix of Y). By central limit theorem we know: $Y \sim \mathcal{N}(0,\Sigma)$

In [36]:
# draw K samples from estimated multivariate normal distribution
m = 2
K = 10000
mu = [0 for i in range(m)]
sample_Y = rng.multivariate_normal(mu, Sigma_hat, K)

In [48]:
# evaluate (1-\alpha) quantile for max within each sample
sample_Y_max = [max(sample_Y[i]) for i in range(len(sample_Y))]

alpha = 0.05
q_alpha_hat = np.quantile(sample_Y_max, 1-alpha) 
print(f"overall max: {np.max(sample_Y_max)}, q_alpha_hat: {q_alpha_hat}")

overall max: 4.135763695768189, q_alpha_hat: 1.9767280422496896


In [50]:
# set radius equal to this quantile, and compute feasibility certificates (probability guarantees)
sqrt_radius = q_alpha_hat

In [51]:
gamma_x1 = p_hat_x1 - (sqrt_radius / math.sqrt(N)) * sigma_hat_x1
print(f"true prob: {p_x1}, empirical prob: {p_hat_x1}, certificate: {gamma_x1}")

true prob: 0.7, empirical prob: 0.65, certificate: 0.5134812592951794


In [52]:
gamma_x2 = p_hat_x2 - (sqrt_radius / math.sqrt(N)) * sigma_hat_x2
print(f"true prob: {p_x2}, empirical prob: {p_hat_x2}, certificate: {gamma_x2}")

true prob: 0.8, empirical prob: 0.85, certificate: 0.7318797195334201
